In [12]:
import pandas as pd
import numpy as np
import sys
import csv

In [15]:
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    "/content/email_dataset_classification.csv",
    engine='python',
    on_bad_lines='skip'
)

In [23]:
print(df.shape)
df.head()

(298166, 3)


,Unnamed: 0,message,label
0,0,['Fabulous discounts from CanadianPharmacy. 50...,1
1,1,http://houston.cowparade.net/,0
2,2,=================================\n\nGuarantee...,1
3,3,Thanks for your help! ---------------------- ...,0
4,4,Start Date: 10/13/01; HourAhead hour: 23; No a...,0


In [46]:
df["message"][2021]

"\nDear Users,\n\nI am new to R, I have just downloaded it to play with. I am a Java  & \nMatlab user, so I assume that it  wouldn't be a hurdle.  Where do I get \nto see the source codes for R  sub-packages  ? I looked at those \nsub-packages in folder  'R-ex'  for the source codes and they are \ncompressed files.  Do I mean to de-compress these files where the source \ncodes reside ?\n\nAny hint would be useful.\n\nCheers,\nSione.\n\n______________________________________________\nR-help@stat.math.ethz.ch mailing list\nhttps://stat.ethz.ch/mailman/listinfo/r-help\nPLEASE do read the posting guide http://www.R-project.org/posting-guide.html\nand provide commented, minimal, self-contained, reproducible code.\n\n"

In [30]:
< > [] \n https link * email id - _

SyntaxError: invalid syntax (3955887718.py, line 1)

In [58]:
import re

def clean_email(text):
    text = text.lower()

    text = re.sub(r'https?\s*:\s*/\s*/\s*\S+', ' URL ', text)
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    text = re.sub(r'[<>\[\]\*]', '', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'[-_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [59]:
df['clean_text'] = df['message'].apply(clean_email)

In [50]:
df.head()

,Unnamed: 0,message,label,clean_text
0,0,['Fabulous discounts from CanadianPharmacy. 50...,1,'fabulous discounts from canadianpharmacy. 50%...
1,1,http://houston.cowparade.net/,0,URL
2,2,=================================\n\nGuarantee...,1,================================= guaranteed t...
3,3,Thanks for your help! ---------------------- ...,0,thanks for your help! tana jones/hou/ect on 09...
4,4,Start Date: 10/13/01; HourAhead hour: 23; No a...,0,start date: 10/13/01; hourahead hour: 23; no a...


In [65]:
for i in range(10):
  print(df["message"][i*1000])
  print("="*50)
  print(df["clean_text"][i*1000])
  print(


  )

['Fabulous discounts from CanadianPharmacy. 50% discounts on all products when you purchase during the summer period. http: / / nationspecial. hk Absolutely downcast prices, caring staff, good and quick shipping, high level of protection, and top quality products are the most common characteristics mentioned by our customers in their recommendation. CanadianPharmacy is proud to be the most popular Canadian drugstore online. Purchase with CanadianPharmacy. Order with CanadianPharmacy and get incredible summer deduction. http: / / nationspecial. hk']
'fabulous discounts from canadianpharmacy. 50% discounts on all products when you purchase during the summer period. URL hk absolutely downcast prices, caring staff, good and quick shipping, high level of protection, and top quality products are the most common characteristics mentioned by our customers in their recommendation. canadianpharmacy is proud to be the most popular canadian drugstore online. purchase with canadianpharmacy. order w

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X = vectorizer.fit_transform(df['clean_text'])

In [67]:
from sklearn.model_selection import train_test_split

y = df['label']   # or your column name

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [70]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB(class_prior=[0.4, 0.6])
model.fit(X_train, y_train)

MultinomialNB(class_prior=[0.4, 0.6])

In [71]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.92      0.95     41707
           1       0.84      0.96      0.90     17927

    accuracy                           0.93     59634
   macro avg       0.91      0.94      0.93     59634
weighted avg       0.94      0.93      0.94     59634



In [72]:
y_prob = model.predict_proba(X_test)[:, 1]

In [74]:
model.predict_proba(X_test[:3])

array([[6.11587423e-05, 9.99938841e-01],
       [5.03222386e-09, 9.99999995e-01],
       [7.48829152e-01, 2.51170848e-01]])